# Step 9: compare available route predictions

Use the ten student-test trips only for final model selection, after model and decoder parameters were frozen using training data. This is a development holdout; the final evaluator partition remains unused.

Compare the untuned ranker, tuned ranker independently, and tuned ranker with HMM/Viterbi. Teammates' logistic regression, KNN, decision tree and stacking are recorded as pending until their real predictions and models are merged. No missing model receives an invented score.

Selection policy fixed before evaluation: retain models within **0.2 percentage points** of the best Top-1, then prefer fewer directed-unreachable consecutive transitions, then lower model-plus-decoder prediction time.

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support

%run 08_hmm_helpers.ipynb

model_dir = Path("results/step7_xgb/tuned_sample10000")
baseline_dir = Path("results/step7_xgb/untuned")
hmm_dir = Path("results/hmm/ranker")
output = Path("results")
assert Path("outputs/step7_xgb_sample10000/best_params.json").exists(), "Finish and freeze Step 7 first."
required = [model_dir / "student_test_predictions.csv", model_dir / "model.json",
            baseline_dir / "student_test_predictions.csv", baseline_dir / "model.json",
            hmm_dir / "decoded_points.csv", hmm_dir / "candidate_probabilities.parquet",
            hmm_dir / "hmm_config.json", hmm_dir / "timings.json"]
for path in required:
    assert path.exists(), f"Required incoming artifact not ready: {path}"
points = pd.read_csv("outputs/student_test/points.csv", dtype={"trip_id": str})
assert points["group_id"].is_unique
valid_points = points[~points["bad_gps"]].copy()
emissions = pd.read_csv(model_dir / "student_test_predictions.csv")
baseline_emissions = pd.read_csv(baseline_dir / "student_test_predictions.csv")
posteriors = pd.read_parquet(hmm_dir / "candidate_probabilities.parquet")
decoded = pd.read_csv(hmm_dir / "decoded_points.csv", dtype={"trip_id": str})
assert not emissions.duplicated(["group_id", "segment_id"]).any()
assert not posteriors.duplicated(["group_id", "segment_id"]).any()
assert not decoded.duplicated("group_id").any()
pd.testing.assert_frame_equal(
    emissions[["group_id", "segment_id"]].sort_values(["group_id", "segment_id"]).reset_index(drop=True),
    posteriors[["group_id", "segment_id"]].sort_values(["group_id", "segment_id"]).reset_index(drop=True),
    check_dtype=False,
)
pd.testing.assert_frame_equal(
    emissions[["group_id", "segment_id"]].sort_values(["group_id", "segment_id"]).reset_index(drop=True),
    baseline_emissions[["group_id", "segment_id"]].sort_values(["group_id", "segment_id"]).reset_index(drop=True),
    check_dtype=False,
)
assert set(decoded["group_id"]) == set(points["group_id"])
point_counts = points.set_index("group_id")["candidate_count"]
assert decoded.loc[decoded["group_id"].map(point_counts).eq(0), "segment_id"].eq(-1).all()
assert decoded.loc[decoded["group_id"].map(point_counts).gt(0), "segment_id"].ge(0).all()
comparison = pd.read_csv("results/step7_xgb/comparison.csv")

## Common scoring

Exclude flagged GPS points from target-based metrics, but keep candidate misses and empty shortlists in the **Top-1 denominator**. Candidate-row precision, recall, macro F1 and the 2×2 confusion matrix use the chosen segment for each point. The HMM uses the Viterbi choice, not posterior argmax.

Log loss is conditional on the true candidate being present. For the HMM it uses actual forward/backward posterior probabilities. It describes marginal probability quality; the Viterbi path need not match each posterior argmax.

Reference labels enter only this scoring section. They never enter emissions, transition geometry or decoding.

In [2]:
def evaluate_predictions(probabilities, selected):
    probabilities = probabilities[["group_id", "segment_id", "score"]].copy()
    assert np.isfinite(probabilities["score"]).all() and probabilities["score"].between(0, 1).all()
    np.testing.assert_allclose(probabilities.groupby("group_id")["score"].sum(), 1, atol=1e-6)
    assert not selected.duplicated("group_id").any()
    selected = selected[selected["segment_id"] >= 0].copy()
    keys = probabilities[["group_id", "segment_id"]]
    assert len(selected.merge(keys, on=["group_id", "segment_id"])) == len(selected)
    rows = probabilities.merge(valid_points[["group_id", "true_arc_id"]], on="group_id", validate="many_to_one")
    rows["label"] = rows["segment_id"].eq(rows["true_arc_id"]).astype("int8")
    chosen = selected[["group_id", "segment_id"]].assign(predicted=1)
    rows = rows.merge(chosen, on=["group_id", "segment_id"], how="left", validate="one_to_one")
    prediction = rows["predicted"].fillna(0).astype("int8")
    precision, recall, f1, _ = precision_recall_fscore_support(
        rows["label"], prediction, labels=[0, 1], zero_division=0)
    positive_scores = rows.loc[rows["label"] == 1, "score"]
    predicted = valid_points.merge(selected[["group_id", "segment_id"]], on="group_id", how="left", validate="one_to_one")
    correct = predicted["segment_id"].eq(predicted["true_arc_id"])
    assert rows.groupby("group_id")["label"].sum().le(1).all()
    return {"top1": float(correct.sum() / len(valid_points)), "macro_f1": float(f1.mean()),
            "precision": float(precision[1]), "recall": float(recall[1]),
            "log_loss": float(-np.log(positive_scores.clip(lower=1e-15)).mean()),
            "candidate_coverage": len(positive_scores) / len(valid_points),
            "valid_points": len(valid_points), "candidate_misses": len(valid_points) - len(positive_scores),
            "confusion_matrix": confusion_matrix(rows["label"], prediction, labels=[0, 1]).tolist()}


def top_candidates(probabilities):
    ordered = probabilities.sort_values(["group_id", "segment_id"]).reset_index(drop=True)
    return ordered.loc[ordered.groupby("group_id")["score"].idxmax(), ["group_id", "segment_id"]].reset_index(drop=True)

## Directed route continuity

A consecutive observation pair is reachable if the selected arc is unchanged or the directed graph has any path from the earlier arc's end node to the later arc's start node. This allows intermediate arcs between GPS observations; an endpoint mismatch alone is **not** called an illegal jump.

Count only adjacent original GPS rows within each trip, including flagged rows because reference quality must not affect decoding. Pairs with missing predictions are reported separately. Directed reachability alone does not prove a route is physically plausible within the elapsed time or short enough; the decoder's transition cost also considers travel distance. No full intermediate path is claimed here.

In [3]:
def route_continuity(selected):
    selected = selected[["group_id", "segment_id"]].copy()
    merged = points[["trip_id", "group_id", "point_id"]].merge(selected, on="group_id", how="left", validate="one_to_one")
    counts = {"transition_pairs": 0, "reachable_pairs": 0, "disconnected_jumps": 0, "missing_transition_pairs": 0}
    for trip_id, frame in merged.groupby("trip_id", sort=True):
        frame = frame.sort_values("point_id")
        network = load_network(trip_id, partition="student_test")
        roads = frame["segment_id"].to_numpy()
        for previous, current in zip(roads[:-1], roads[1:]):
            if not np.isfinite(previous) or not np.isfinite(current) or min(previous, current) < 0:
                counts["missing_transition_pairs"] += 1
                continue
            counts["transition_pairs"] += 1
            reachable = transition_is_reachable(network, int(previous), int(current))
            counts["reachable_pairs"] += int(reachable)
            counts["disconnected_jumps"] += int(not reachable)
    counts["route_continuity"] = counts["reachable_pairs"] / counts["transition_pairs"] if counts["transition_pairs"] else None
    return counts

## Evaluate and select

Prediction time is the saved model prediction plus optional network/decoder time on **precomputed features**, over all ten trips. It excludes raw-file candidate generation and feature engineering, so it is not an end-to-end API latency claim.

Both rankers use the same feature list. The HMM uses only the tuned ranker; there is no extra baseline-HMM fit. Decoder parameters must come from training OOF data. Missing-prediction pairs cannot improve selection: unsupported transitions include both disconnected jumps and missing pairs.

In [4]:
ranker_row = comparison[comparison["variant"] == "tuned_sample10000"].iloc[0]
model_seconds = float(ranker_row["final_predict_s"])
baseline_seconds = float(comparison.loc[comparison["variant"] == "untuned", "final_predict_s"].iloc[0])
decode_timings = json.loads((hmm_dir / "timings.json").read_text())
decode_seconds = float(decode_timings["decoder_predict_s"])
selections = {"ranker_untuned": top_candidates(baseline_emissions), "ranker": top_candidates(emissions), "ranker_hmm": decoded[["group_id", "segment_id"]]}
probabilities = {"ranker_untuned": baseline_emissions, "ranker": emissions, "ranker_hmm": posteriors}
records = []
for name in ["ranker_untuned", "ranker", "ranker_hmm"]:
    metrics = evaluate_predictions(probabilities[name], selections[name])
    continuity = route_continuity(selections[name])
    seconds = baseline_seconds if name == "ranker_untuned" else model_seconds + (decode_seconds if name.endswith("hmm") else 0)
    records.append({"model": name, **metrics, **continuity, "prediction_s": seconds,
                    "prediction_ms_per_trip": 1000 * seconds / points["trip_id"].nunique(),
                    "timing_scope": "model plus optional decoder on precomputed features; excludes candidates/features"})
final = pd.DataFrame(records)
final["unsupported_transitions"] = final["disconnected_jumps"] + final["missing_transition_pairs"]
final.to_csv(output / "final_comparison.csv", index=False)
near_best = final[final["top1"] >= final["top1"].max() - 0.002]
winner = near_best.sort_values(["unsupported_transitions", "prediction_s", "model"]).iloc[0]
selected_dir = baseline_dir if winner["model"] == "ranker_untuned" else model_dir
selection = {"selected_name": winner["model"], "model_path": str(selected_dir / "model.json"),
             "model_dir": str(selected_dir), "feature_list_path": "outputs/folds/contract.json",
             "hmm_enabled": winner["model"] == "ranker_hmm",
             "hmm_config_path": str(hmm_dir / "hmm_config.json") if winner["model"] == "ranker_hmm" else None,
             "comparison_path": str(output / "final_comparison.csv"),
             "rationale": "Within 0.2 percentage points of best student-test Top-1, then fewest disconnected or missing adjacent transitions, then shortest prediction time.",
             "metrics": {key: value.item() if isinstance(value, np.generic) else value for key, value in winner.to_dict().items()},
             "timing_scope": "precomputed-feature model and optional decoder time; not complete API latency",
             "scope": "available ranker variants only; stacking and teammate models pending merge",
             "selection_dataset": "student-test development holdout; final evaluator trips unused"}
(output / "final_selection.json").write_text(json.dumps(selection, indent=2, allow_nan=False) + "\n")
status = {"logistic_regression": "pending model and student-test predictions",
          "knn": "pending model and student-test predictions", "decision_tree": "pending model and student-test predictions",
          "stacking": "pending base-model OOF merge, combiner training and student-test predictions"}
(output / "pending_model_comparisons.json").write_text(json.dumps(status, indent=2) + "\n")
display(final[["model", "top1", "macro_f1", "log_loss", "route_continuity", "disconnected_jumps", "missing_transition_pairs", "prediction_s"]])
print("Selected available model:", winner["model"])

,model,top1,macro_f1,log_loss,route_continuity,disconnected_jumps,missing_transition_pairs,prediction_s
0,ranker_untuned,0.888413,0.943291,0.302737,0.999321,13,0,0.832187
1,ranker,0.891493,0.944856,0.310644,0.999530,9,0,0.515766
2,ranker_hmm,0.885960,0.942044,0.685108,1.000000,0,0,72.576762


Selected available model: ranker
